# DeepFake Detection — Google Colab Training

**Architecture:** DINOv2 ViT-B/14 + FFT/DCT LightCNN + Temperature Scaling

**Stratégie stockage :**
- `data/` → Google Drive (permanent — datasets trop lourds pour /content/ éphémère)
- `checkpoints/`, `outputs/`, `logs/` → Google Drive (permanent)

| Étape | Cellule |
|-------|--------|
| Vérifier GPU | §1 |
| Monter Drive | §2 |
| Cloner repo | §3 |
| Installer dépendances | §4 |
| Lier Drive ↔ repo | §5 |
| Télécharger images réelles (FF++ + CelebDF) | §6a |
| Télécharger fakes DF40 (4 méthodes) | §6b |
| Vérifier structure dataset | §6c |
| Configurer l'entraînement | §7 |
| **Entraîner (test 5 epochs)** | §8 |
| **Entraîner (full 30 epochs)** | §8b |
| Évaluer | §9 |
| Inférence | §10 |
| Afficher résultats | §11 |
| Télécharger modèle | §12 |
| Anti-idle | §13 |

## 1. Vérifier GPU

In [ ]:
import torch, shutil

if not torch.cuda.is_available():
    raise RuntimeError("GPU non détecté !\n→ Runtime > Modifier le type d'exécution > T4 GPU")

print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

total, used, free = shutil.disk_usage('/content')
print(f"Disque local libre : {free/1e9:.0f} GB")

## 2. Monter Google Drive

In [ ]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/DeepFakeDetection'

# Tous les dossiers vont sur Drive (données + checkpoints)
for folder in ['data/df40/fake', 'data/real', 'checkpoints', 'outputs', 'logs']:
    os.makedirs(f'{DRIVE_ROOT}/{folder}', exist_ok=True)

print(f"Drive monté : {DRIVE_ROOT}")
_, _, free = shutil.disk_usage(DRIVE_ROOT)
print(f"Drive libre : {free/1e9:.1f} GB")

## 3. Cloner / Mettre à jour le repo GitHub

In [ ]:
import os

REPO_URL = 'https://github.com/anessrb/Deepfakedetection.git'
REPO_DIR = '/content/Deepfakedetection'

if os.path.exists(f'{REPO_DIR}/.git'):
    print("Repo déjà cloné — mise à jour...")
    !cd {REPO_DIR} && git pull --quiet
else:
    print("Clonage du repo...")
    !git clone --quiet {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Répertoire : {os.getcwd()}")
!git log --oneline -3

## 4. Installer les dépendances

In [ ]:
!pip install -q -r requirements.txt

import torch, timm, albumentations, cv2
print(f"torch         : {torch.__version__}")
print(f"timm          : {timm.__version__}")
print(f"albumentations: {albumentations.__version__}")
print(f"opencv        : {cv2.__version__}")
print("OK.")

## 5. Lier Drive ↔ Repo (symlinks)

Tous les dossiers pointent vers Drive — tout est permanent.

In [ ]:
import os

REPO_DIR   = '/content/Deepfakedetection'
DRIVE_ROOT = '/content/drive/MyDrive/DeepFakeDetection'

def make_symlink(link, target):
    if os.path.islink(link):
        os.remove(link)
    elif os.path.isdir(link) and not os.listdir(link):
        os.rmdir(link)
    if not os.path.exists(link):
        os.symlink(target, link)
        return True
    return False

# data/ et tous les sous-dossiers → Drive
for folder in ['data', 'checkpoints', 'outputs', 'logs']:
    target = f'{DRIVE_ROOT}/{folder}'
    os.makedirs(target, exist_ok=True)
    status = '✓' if make_symlink(f'{REPO_DIR}/{folder}', target) else '~'
    print(f"  {status} {folder}/ → Drive")

print("\nContenu data/ :", os.listdir(f'{REPO_DIR}/data') or '(vide)')

## 6a. Télécharger les images réelles (FF++ + CelebDF)

Source : liens officiels DF40 README  
Sauvegardé sur Drive → **permanent**, pas besoin de re-télécharger.

In [ ]:
import os
from pathlib import Path
!pip install -q gdown

real_dir = Path('data/real')
real_dir.mkdir(parents=True, exist_ok=True)

REAL_SOURCES = {
    # Liens officiels du README DF40
    'ffpp_real'   : '1dHJdS0NZ6wpewbGA5B0PdIBS9gz28pdb',  # FF++ real frames
    'celebdf_real': '1FGZ3aYsF-Yru50rPLoT5ef8-2Nkt4uBw',  # CelebDF real frames
}

for name, file_id in REAL_SOURCES.items():
    out_dir  = real_dir / name
    zip_path = real_dir / f'{name}.zip'

    if out_dir.exists() and any(out_dir.rglob('*.jpg')) or any(out_dir.rglob('*.png')) if out_dir.exists() else False:
        n = sum(1 for _ in out_dir.rglob('*.jpg')) + sum(1 for _ in out_dir.rglob('*.png'))
        print(f"  ~ {name}: déjà présent ({n:,} images)")
        continue

    print(f"  ↓ {name}...")
    !gdown {file_id} -O {str(zip_path)} --fuzzy

    if zip_path.exists():
        out_dir.mkdir(exist_ok=True)
        !unzip -q {str(zip_path)} -d {str(out_dir)}/
        zip_path.unlink()
        n = sum(1 for _ in out_dir.rglob('*.jpg')) + sum(1 for _ in out_dir.rglob('*.png'))
        print(f"    ✓ {name}: {n:,} images")
    else:
        print(f"    ✗ Échec {name} — vérifie les permissions Drive")

# Résumé
total_real = sum(1 for _ in real_dir.rglob('*.jpg')) + sum(1 for _ in real_dir.rglob('*.png'))
print(f"\nTotal images réelles : {total_real:,}")

## 6b. Télécharger fakes DF40 — 4 méthodes

Structure réelle DF40 après extraction :
```
data/df40/fake/
├── simswap/ff/    ← domaine FF++
├── simswap/cdf/   ← domaine CelebDF
├── fomm/ff/
└── ...
```
Sauvegardé sur Drive → **permanent**.

In [ ]:
import os
from pathlib import Path

fake_dir = Path('data/df40/fake')
fake_dir.mkdir(parents=True, exist_ok=True)

# 4 méthodes représentatives — IDs Google Drive officiels DF40
# (liens publics du README : https://github.com/YZY-stack/DF40)
METHODS = {
    'simswap'  : '1vnEXjxgSxmiNY-RkLQdsbhayTvAAoOIc',  # face-swap
    'fomm'     : '1UgGDvGGw5H6Wf0KTHzjKoigHB5ALcC5Q',  # reenactment
    'sadtalker': '1DQCVDlFInuAH3ryQgZIyKQzPiEIyFaaa',  # reenactment audio-driven
    'inswap'   : '1hEsN-oY9Ye2OiAzGn03UD7AajztZ6b_B',  # face-swap
}

for name, file_id in METHODS.items():
    out_dir  = fake_dir / name
    zip_path = fake_dir / f'{name}.zip'

    if out_dir.exists() and (any(out_dir.rglob('*.jpg')) or any(out_dir.rglob('*.png'))):
        n = sum(1 for _ in out_dir.rglob('*.jpg')) + sum(1 for _ in out_dir.rglob('*.png'))
        print(f"  ~ {name:<12}: déjà présent ({n:,} images)")
        continue

    print(f"  ↓ {name}...")
    !gdown {file_id} -O {str(zip_path)} --fuzzy

    if zip_path.exists():
        out_dir.mkdir(exist_ok=True)
        !unzip -q {str(zip_path)} -d {str(out_dir)}/
        zip_path.unlink()
        n = sum(1 for _ in out_dir.rglob('*.jpg')) + sum(1 for _ in out_dir.rglob('*.png'))
        print(f"    ✓ {name}: {n:,} images")
    else:
        print(f"    ✗ Échec {name}")

import shutil
_, _, free = shutil.disk_usage('/content/drive/MyDrive')
print(f"\nDrive libre restant : {free/1e9:.1f} GB")

## 6c. Vérifier la structure et créer le dataset unifié

Crée `data/df40/real/` en symlinkant les images réelles, et vérifie que tout est en place.

In [ ]:
from pathlib import Path

base     = Path('data/df40')
real_src = Path('data/real')
fake_dir = base / 'fake'

# Drive ne supporte pas les symlinks — le loader utilise data/real/ en fallback automatiquement
print("=== Structure data/df40/ ===")
n_real = sum(1 for _ in real_src.rglob('*.jpg')) + sum(1 for _ in real_src.rglob('*.png')) if real_src.exists() else 0
print(f"  real/  (data/real/) : {n_real:,} images")

n_fake_total = 0
if fake_dir.exists():
    for m in sorted(fake_dir.iterdir()):
        if m.is_dir():
            n = sum(1 for _ in m.rglob('*.jpg')) + sum(1 for _ in m.rglob('*.png'))
            n_fake_total += n
            print(f"  fake/{m.name:<12}: {n:,} images")

print(f"  fake/ TOTAL  : {n_fake_total:,} images")
print(f"  GRAND TOTAL  : {n_real + n_fake_total:,} images")

if n_real == 0:
    print("\n⚠ Pas d'images réelles — lance §6a")
if n_fake_total == 0:
    print("\n⚠ Pas de fakes — lance §6b")
if n_real > 0 and n_fake_total > 0:
    print("\n✓ Dataset prêt pour l'entraînement !")

## 7. Configurer l'entraînement pour Colab

In [ ]:
import yaml, torch

with open('configs/default.yaml', 'r') as f:
    config = yaml.safe_load(f)

vram_gb    = torch.cuda.get_device_properties(0).total_memory / 1e9
batch_size = 64 if vram_gb >= 30 else 32

config['training']['batch_size']              = batch_size
config['training']['num_workers']             = 2
config['training']['epochs']                  = 30
config['training']['use_amp']                 = True
config['training']['checkpoint_dir']          = 'checkpoints/'
config['datasets']['data_root']               = 'data/'
config['datasets']['max_samples_per_dataset'] = None
config['evaluation']['cross_dataset_eval']    = ['df40']

# Pointer le df40 root vers la bonne structure
config['datasets']['df40']['root'] = 'data/df40/'

with open('configs/colab.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("Config : configs/colab.yaml")
print(f"  GPU        : {torch.cuda.get_device_name(0)} ({vram_gb:.0f} GB)")
print(f"  batch_size : {batch_size} | epochs : 30 | AMP : True")
print(f"  data/df40/ → Drive (permanent)")

## 8. Test rapide — 5 epochs (~20-30 min)

Vérifie que tout le pipeline fonctionne avant de lancer le vrai entraînement.

In [ ]:
!python scripts/train.py \
    --config configs/colab.yaml \
    --epochs 5 \
    --output_dir outputs/test_run/

## 8b. Entraînement complet — 30 epochs (~3-4h)

Lance §13 (anti-idle) dans un onglet séparé avant de démarrer.

In [ ]:
!python scripts/train.py \
    --config configs/colab.yaml \
    --output_dir outputs/

In [ ]:
# Reprendre après déconnexion
# !python scripts/train.py \
#     --config configs/colab.yaml \
#     --resume checkpoints/best.pth \
#     --output_dir outputs/

## 9. Évaluation

In [ ]:
import os

for ckpt in ['outputs/detector_calibrated.pth', 'checkpoints/best.pth']:
    if os.path.exists(ckpt):
        print(f"Checkpoint : {ckpt}")
        !python scripts/evaluate.py \
            --checkpoint {ckpt} \
            --output_dir outputs/evaluation/ \
            --config configs/colab.yaml
        break
else:
    print("Aucun checkpoint. Lance §8 d'abord.")

## 10. Inférence

In [ ]:
from google.colab import files
import os

print("Upload une image (.jpg/.png) ou vidéo (.mp4) :")
uploaded = files.upload()

ckpt = 'outputs/detector_calibrated.pth'
if not os.path.exists(ckpt):
    ckpt = 'checkpoints/best.pth'

for fname in uploaded.keys():
    print(f"\n→ Inférence : {fname}")
    !python scripts/inference.py \
        --input "{fname}" \
        --checkpoint {ckpt} \
        --output_dir outputs/inference/

## 11. Afficher les résultats

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

plots = [
    ('Training History',  'outputs/training_history.png'),
    ('Calibration Curve', 'outputs/calibration_curve.png'),
]

available = [(t, p) for t, p in plots if Path(p).exists()]
if not available:
    print("Aucun graphique. Lance l'entraînement d'abord.")
else:
    fig, axes = plt.subplots(1, len(available), figsize=(7 * len(available), 5))
    if len(available) == 1:
        axes = [axes]
    for ax, (title, path) in zip(axes, available):
        ax.imshow(mpimg.imread(path))
        ax.set_title(title, fontsize=12)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 12. Télécharger le modèle

In [ ]:
from google.colab import files
import os

for path in ['outputs/detector_calibrated.pth', 'checkpoints/best.pth']:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f"Téléchargement : {path} ({size_mb:.0f} MB)")
        files.download(path)
        break
else:
    print("Aucun modèle. Lance §8 d'abord.")

## 13. Anti-idle — éviter la déconnexion Colab

Lance dans un onglet séparé pendant l'entraînement.

In [ ]:
import time
from IPython.display import clear_output

for i in range(9999):
    time.sleep(60)
    clear_output(wait=True)
    print(f"Session active — {i+1} min écoulées. (■ pour arrêter)")